In [10]:
'''
# Live Pipeline Test — AI Support Agent

Runs the actual agent end-to-end on a sample message: classify → retrieve →
draft reply → escalation decision. Requires a Groq API key (free tier at
console.groq.com). Uses a 1,000-thread sample corpus for retrieval, not the
full dataset, to keep this notebook lightweight — see `01_development_and_exploration.ipynb`
for the full 10,428-thread pipeline.
'''

'\n# Live Pipeline Test — AI Support Agent\n\nRuns the actual agent end-to-end on a sample message: classify → retrieve →\ndraft reply → escalation decision. Requires a Groq API key (free tier at\nconsole.groq.com). Uses a 1,000-thread sample corpus for retrieval, not the\nfull dataset, to keep this notebook lightweight — see `01_development_and_exploration.ipynb`\nfor the full 10,428-thread pipeline.\n'

In [11]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

In [12]:
!pip install -q groq sentence-transformers faiss-cpu

In [13]:
import os
os.environ['GROQ_API_KEY'] = input("Paste your Groq API key: ")

In [14]:
import pickle
from groq import Groq

INTENT_TAXONOMY = {
    "flight_disruption_refund": "Delays, cancellations, missed connections, or requests for refunds/compensation due to a disrupted flight.",
    "booking_reservation": "Issues with booking confirmations, reservation references, name changes, or reservation modifications.",
    "baggage_fees": "Questions or complaints about baggage handling, checked-bag fees, or other ancillary fees.",
    "policy_information": "General questions about airline policy, rules, or pricing that are not tied to a specific disrupted trip (e.g., carry-on rules, upgrade eligibility).",
    "service_quality_complaint": "Complaints about staff behavior, rudeness, poor treatment, or onboard discomfort, often seeking acknowledgment or compensation.",
    "technical_app_issue": "Problems with the airline's app, website, or digital check-in/booking systems.",
    "positive_feedback": "Compliments, gratitude, or positive comments with no actionable request.",
    "non_actionable_other": "Sarcasm, off-topic content, out-of-scope requests, or abusive/toxic language not requiring a substantive support response.",
}

with open('../data/retrieval_demo_corpus.pkl', 'rb') as f:
    resolvable_threads = pickle.load(f)

client = Groq()
print(f"Loaded {len(resolvable_threads)} threads for retrieval demo")

Loaded 1000 threads for retrieval demo


In [15]:
def format_thread(thread):
    return "\n".join(f"{'Customer' if t['inbound'] else 'Brand'}: {t['text']}" for t in thread)

def classify_intent(thread_text):
    taxonomy_text = "\n".join(f"- {k}: {v}" for k, v in INTENT_TAXONOMY.items())
    prompt = f"""Classify the CUSTOMER'S PRIMARY INTENT into exactly one category:
{taxonomy_text}

Message: {thread_text}

Respond with ONLY the category key."""
    r = client.chat.completions.create(model="openai/gpt-oss-20b", messages=[{"role": "user", "content": prompt}],
                                        temperature=0, max_tokens=200, reasoning_effort="low")
    label = r.choices[0].message.content.strip()
    return label if label in INTENT_TAXONOMY else "PARSE_ERROR"

import faiss
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
openers = [t[0]['text'] for t in resolvable_threads]
embeddings = embed_model.encode(openers, show_progress_bar=True)
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype('float32'))

def retrieve_similar(query, k=3, threshold=0.65):
    qv = embed_model.encode([query])
    faiss.normalize_L2(qv)
    scores, idx = index.search(qv.astype('float32'), k)
    return [{"score": float(s), "thread": resolvable_threads[i]} for s, i in zip(scores[0], idx[0]) if s >= threshold]

def draft_reply(message, k=3):
    retrieved = retrieve_similar(message, k=k)
    if not retrieved:
        return None, "No sufficiently similar precedent found."
    examples = "\n".join(f"--- (sim {r['score']:.2f}) ---\n{format_thread(r['thread'])}\n" for r in retrieved)
    prompt = f"""You are a customer support agent for American Airlines.
Similar past resolutions:
{examples}
Draft a brief, empathetic reply. Do not invent specific facts not supported by the examples.
New customer message: "{message}"
Draft reply:"""
    r = client.chat.completions.create(model="openai/gpt-oss-20b", messages=[{"role": "user", "content": prompt}],
                                        temperature=0.3, max_tokens=300, reasoning_effort="low")
    return r.choices[0].message.content.strip(), retrieved

ESCALATION_RULES = {"flight_disruption_refund": "escalate", "service_quality_complaint": "escalate",
                     "non_actionable_other": "escalate", "booking_reservation": "auto_handle",
                     "baggage_fees": "auto_handle", "policy_information": "auto_handle",
                     "technical_app_issue": "auto_handle", "positive_feedback": "auto_handle"}
ESCALATION_KEYWORDS = ["lawsuit", "lawyer", "sue", "legal action", "legal", "discriminat", "injury", "injured", "unsafe", "emergency", "attorney"]

def decide_escalation(intent, message):
    matched = [k for k in ESCALATION_KEYWORDS if k in message.lower()]
    if matched:
        return "escalate", f"Keyword override: matched {matched}"
    return ESCALATION_RULES.get(intent, "escalate"), f"Based on intent category: {intent}"

print("Pipeline functions loaded.")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Pipeline functions loaded.


In [16]:
test_message = "My flight got delayed 3 hours and I want a refund"  # <-- change this to try your own

intent = classify_intent(test_message)
reply, sources = draft_reply(test_message)
decision, reason = decide_escalation(intent, test_message)

print(f"CUSTOMER MESSAGE:\n{test_message}\n")
print(f"1. PREDICTED INTENT: {intent}\n")
print(f"2. DRAFTED REPLY:\n{reply}\n")
print(f"3. ESCALATION DECISION: {decision}\nREASON: {reason}")

CUSTOMER MESSAGE:
My flight got delayed 3 hours and I want a refund

1. PREDICTED INTENT: flight_disruption_refund

2. DRAFTED REPLY:
None

3. ESCALATION DECISION: escalate
REASON: Based on intent category: flight_disruption_refund


In [18]:
# Debug: see the raw retrieval scores before the threshold filter
raw_matches = retrieve_similar(test_message, k=3, threshold=0.0)  # threshold=0 shows everything
for m in raw_matches:
    print(f"Score: {m['score']:.3f}")
    print(format_thread(m['thread'])[:150], "...\n")

Score: 0.608
Customer: @AmericanAir On a work trip and got to the airport to be told my flight is delayed 4 hours. Waited in a ticket agent line for 30- it did not ...

Score: 0.593
Customer: Emailed @AmericanAir on Monday regarding a refund for a cancelled flight and still no response https://t.co/xUl8WfT6KI
Brand: @328107 Please ...

Score: 0.587
Customer: @AmericanAir need help. Booked another flight on another airline after you guys changed my flight without notice. Now phone rep won't refund ...



In [17]:
test_message_2 = "Your staff was so rude I'm considering legal action"
intent_2 = classify_intent(test_message_2)
decision_2, reason_2 = decide_escalation(intent_2, test_message_2)
print(f"Message: {test_message_2}")
print(f"Predicted intent: {intent_2} -> Decision: {decision_2}")
print(f"Reason: {reason_2}")

Message: Your staff was so rude I'm considering legal action
Predicted intent: service_quality_complaint -> Decision: escalate
Reason: Keyword override: matched ['legal action', 'legal']
